# Date of planet conjunctions: when do five or more planets meet?

Ported from the Colab `Date_Planet_Conjuction.ipynb` (archived repository `astro-fusion/Astrology_Planetary_Data`). It finds the major dates when five or more planets are in one sign, plots events per year and tabulates the largest.

> **Sidereal correction.** The Colab originals read
> `Astrology_Planetary_Data` (now archived), whose CSVs hold only house
> occupancy per day for an unsourced ascendant and could not be reproduced
> with either a tropical or a Lahiri-sidereal calculation. This version reads
> the regenerated dataset (`libs/vedic_astrology_core/datasets`): sidereal
> Lahiri, mean node, Ketu = Rahu + 180, whole-sign. In whole-sign charts
> "same house" is the same as "same sign", so the sign is used and no
> ascendant is needed. Rahu and Ketu are always 180 degrees apart, so they
> never share a sign.

In [ ]:
# Parameters (papermill overrides these)
START_YEAR = 1600
END_YEAR = 2100
DATA_DIR = "../data/datasets/planetary"
MIN_PLANETS = 5   # planets in one sign
CLUSTER_DAYS = 3  # dates within this many days collapse to one event

## Load the dataset

In [ ]:
from pathlib import Path

import pandas as pd

from vedic_astrology_core.datasets.planetary import combined, generate

years = range(START_YEAR, END_YEAR + 1)
try:
    df = combined(years, data_dir=DATA_DIR)
except FileNotFoundError:
    # Regenerate offline with the verified engine (about 1 s per 3 years),
    # or download a century zip from the release page:
    # https://github.com/astro-fusion/astro_research-white-paper/releases
    generate(START_YEAR, END_YEAR, out_dir=DATA_DIR, fmt="parquet")
    df = combined(years, data_dir=DATA_DIR)
df["day"] = (df["jd"] + 0.5).astype(int)  # integer Julian day number
df["year"] = df["date"].str[:4].astype(int)
print(df.shape)
df.head()

## Dates with a crowded sign

In [ ]:
signs = (
    df.groupby(["day", "year", "sign", "sign_name"])
    .agg(n=("body", "size"), bodies=("body", lambda s: " ".join(s)))
    .reset_index()
)
crowded = signs[signs["n"] >= MIN_PLANETS].copy()
print(f"{len(crowded)} (day, sign) rows with >= {MIN_PLANETS} planets")
crowded.head(10)

In [ ]:
import numpy as np


def cluster_days(days, gap):
    """Group sorted integer days whose neighbours are at most ``gap`` apart.

    Returns one dict per cluster with its first, last and central day.
    """
    days = np.sort(np.unique(days))
    if len(days) == 0:
        return []
    breaks = np.where(np.diff(days) > gap)[0] + 1
    return [
        {"first": g[0], "last": g[-1], "centre": int(np.median(g)), "n": len(g)}
        for g in np.split(days, breaks)
    ]


def day_to_date(day):
    """ISO date of an integer Julian day number."""
    from vedic_astrology_core.utils.datetime_utils import julian_day_to_datetime

    return julian_day_to_datetime(day - 0.5).date().isoformat()

## Group dates within +/- 3 days into one central date

In [ ]:
crowded = crowded.sort_values("day")
events = pd.DataFrame(cluster_days(crowded["day"].to_numpy(), CLUSTER_DAYS))
if len(events):
    peak = crowded.groupby("day")["n"].max()
    events["max_planets"] = [
        int(peak.loc[e.first : e.last].max()) for e in events.itertuples()
    ]
    events["date"] = events["centre"].map(day_to_date)
    events["year"] = events["date"].str[:4].astype(int)
events.head(10)

## Events per year

In [ ]:
import matplotlib.pyplot as plt

counts = (
    events.groupby("year").size().rename("events").reset_index()
    if len(events)
    else pd.DataFrame({"year": [], "events": []})
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(counts["year"], counts["events"], "o")
ax.set_xlabel("Year")
ax.set_ylabel(f"Events with {MIN_PLANETS}+ planets in one sign")
ax.set_title("Planet conjunction events per year (sidereal Lahiri)")
ax.grid(True, linestyle="--", alpha=0.7)
plt.show()

## Largest events

In [ ]:
largest = events.sort_values(["max_planets", "n"], ascending=False).head(10) if len(events) else events
largest[["date", "max_planets", "n"]].rename(columns={"n": "days"}) if len(events) else largest